In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

In [2]:
# Определяем корневую папку проекта.
# Если ноутбук запущен из папки notebooks, поднимаемся на один уровень выше.
current_directory = Path.cwd()

if current_directory.name == "notebooks":
    project_root = current_directory.parent
else:
    project_root = current_directory


# Указываем путь к таблице, подготовленной на шестом этапе.
metrics_path = (
    project_root
    / "reports"
    / "results"
    / "analysis"
    / "retrieval_comparison.csv"
)

In [4]:
# Загружаем результаты сравнения стратегий поиска.
metrics_table = pd.read_csv(metrics_path)


# Заменяем названия стратегий на более понятные.
strategy_names = {
    "bm25_document": "BM25 — целые документы",
    "dense_document": "Dense/BGE — целые документы",
    "hybrid_document": "Гибридный поиск",
    "chunk_fixed": "Фиксированные чанки",
    "chunk_recursive": "Рекурсивные чанки",
    "chunk_sentence": "Чанки по предложениям",
}

metrics_table["method"] = (
    metrics_table["method"]
    .replace(strategy_names)
)


# Переименовываем столбцы для красивого вывода в ноутбуке.
metrics_table = metrics_table.rename(
    columns={
        "method": "Стратегия",
        "recall_at_1": "Recall@1",
        "recall_at_10": "Recall@10",
        "mrr_at_10": "MRR@10",
        "ndcg_at_10": "nDCG@10",
        "recall_at_100": "Recall@100",
    }
)


# Перечисляем столбцы с числовыми метриками.
metric_columns = [
    "Recall@1",
    "Recall@10",
    "MRR@10",
    "nDCG@10",
    "Recall@100",
]


# Выводим отформатированную таблицу.
# Все значения округляются до четырёх знаков после запятой.
# Лучший результат в каждом столбце выделяется зелёным цветом.
display(
    metrics_table.style
    .format({column: "{:.4f}" for column in metric_columns})
    .highlight_max(
        subset=metric_columns,
        color="#b7e1cd",
    )
    .hide(axis="index")
)

Стратегия,Recall@1,Recall@10,MRR@10,nDCG@10,Recall@100
BM25 — целые документы,0.5282,0.8173,0.6464,0.6826,0.9282
Фиксированные чанки,0.5701,0.8386,0.6802,0.7143,0.9517
Рекурсивные чанки,0.5984,0.8340,0.6951,0.7249,0.9517
Чанки по предложениям,0.5859,0.8336,0.6882,0.7178,0.9550
Dense/BGE — целые документы,0.5759,0.8362,0.6822,0.7127,0.9417
Гибридный поиск,0.5718,0.8612,0.6910,0.7268,0.9683


## Запросы и ответы генеративной модели

Для проверки RAG-системы были выбраны восемь утверждений из тестовой части SciFact.

Выборка формировалась случайно, но воспроизводимо (`random_seed = 42`), без повторений:

- четыре запроса имеют один эталонный релевантный документ;
- четыре запроса имеют несколько релевантных документов;
- запросы относятся к разным научным темам;
- в выборке представлены как простые, так и более сложные случаи поиска доказательств.

Такой подход позволяет проверить, как система работает с разным количеством эталонных
источников, и не выбирать вручную только удачные примеры. При этом восемь запросов — это
демонстрационная, а не статистически полная оценка.

Для формирования контекста использовались четыре результата рекурсивного чанкинга.
Рекурсивный чанкинг показал лучший `nDCG@10` среди трёх исследованных вариантов чанкинга.
Ограничение в четыре источника позволяет сохранить контекст достаточно компактным.

Важно: SciFact содержит эталонную разметку на уровне документов, но не на уровне отдельных
чанков. Поэтому мы не можем сравнить «эталонный чанк — найденный чанк» напрямую. Вместо этого
проверяется, относится ли найденный чанк к документу, который SciFact считает релевантным.

### Метрики соответствия найденного контекста эталону

Для каждого запроса выводятся:

- **Retrieval score** — семантическое сходство запроса и чанка по оценке Dense/BGE-поиска;
- **Hit@4** — присутствует ли среди четырёх источников хотя бы один чанк из эталонного документа;
- **Recall@4** — какая доля эталонных документов представлена в найденном контексте;
- **RR@4** — обратная позиция первого найденного эталонного документа;
- **nDCG@4** — насколько высоко расположены релевантные документы;
- **доля подходящих чанков** — доля чанков, родительские документы которых входят в эталонную разметку.

Дополнительно выводятся результаты ручной оценки ответа: корректность, обоснованность,
полнота и качество цитирования. Каждая из этих характеристик оценивалась по шкале от 0 до 2.

In [6]:
from pathlib import Path
from html import escape
import json
import math

import pandas as pd
from IPython.display import HTML, Markdown, display


# Определяем корневую папку проекта.
# Код будет работать как при запуске из корня проекта,
# так и при запуске ноутбука из папки notebooks.
current_directory = Path.cwd()

if current_directory.name == "notebooks":
    project_root = current_directory.parent
else:
    project_root = current_directory


# Файл содержит точные prompts, отправленные GigaChat,
# полученные ответы и использованные источники.
answers_path = (
    project_root
    / "reports"
    / "results"
    / "rag"
    / "stage_5_answers.jsonl"
)


# Файл содержит результаты ручной оценки ответов.
evaluation_path = (
    project_root
    / "reports"
    / "results"
    / "analysis"
    / "rag_evaluation.csv"
)


# Загружаем сохранённые результаты генерации.
with answers_path.open(encoding="utf-8") as file:
    answers = [
        json.loads(line)
        for line in file
        if line.strip()
    ]


# Загружаем таблицу ручной оценки.
evaluation = pd.read_csv(
    evaluation_path,
    dtype={"query_id": str},
)

evaluation_by_query = evaluation.set_index("query_id")


def calculate_context_metrics(answer_record):
    """
    Рассчитывает соответствие найденных чанков эталонным документам SciFact.

    Эталонная разметка существует на уровне документов.
    Поэтому релевантность чанка определяется через doc_id его родительского документа.
    """

    gold_documents = {
        str(doc_id)
        for doc_id in answer_record.get("relevant_doc_ids", [])
    }

    # Располагаем источники в том порядке, в котором они вошли в prompt.
    sources = sorted(
        answer_record.get("sources", []),
        key=lambda source: source.get(
            "retrieval_rank",
            source.get("source_number", 0),
        ),
    )

    source_document_ids = [
        str(source.get("doc_id"))
        for source in sources
    ]

    # Удаляем повторяющиеся документы, сохраняя порядок поиска.
    ranked_unique_documents = list(
        dict.fromkeys(source_document_ids)
    )

    matched_documents = (
        gold_documents.intersection(ranked_unique_documents)
    )

    # Hit показывает, найден ли хотя бы один эталонный документ.
    hit = int(bool(matched_documents))

    # Recall показывает, какая доля всех эталонных документов попала в контекст.
    recall = (
        len(matched_documents) / len(gold_documents)
        if gold_documents
        else 0.0
    )

    # Находим позицию первого эталонного документа.
    first_relevant_rank = next(
        (
            rank
            for rank, doc_id in enumerate(
                ranked_unique_documents,
                start=1,
            )
            if doc_id in gold_documents
        ),
        None,
    )

    reciprocal_rank = (
        1 / first_relevant_rank
        if first_relevant_rank is not None
        else 0.0
    )

    # Рассчитываем nDCG для найденного контекста.
    relevance_values = [
        int(doc_id in gold_documents)
        for doc_id in ranked_unique_documents
    ]

    dcg = sum(
        relevance / math.log2(rank + 1)
        for rank, relevance in enumerate(
            relevance_values,
            start=1,
        )
    )

    ideal_relevant_count = min(
        len(gold_documents),
        len(ranked_unique_documents),
    )

    ideal_dcg = sum(
        1 / math.log2(rank + 1)
        for rank in range(1, ideal_relevant_count + 1)
    )

    ndcg = dcg / ideal_dcg if ideal_dcg else 0.0

    # Отдельно считаем долю чанков, взятых из эталонных документов.
    matched_chunk_count = sum(
        doc_id in gold_documents
        for doc_id in source_document_ids
    )

    chunk_match_rate = (
        matched_chunk_count / len(sources)
        if sources
        else 0.0
    )

    return {
        "gold_documents": sorted(gold_documents),
        "sources": sources,
        "source_document_ids": source_document_ids,
        "matched_documents": sorted(matched_documents),
        "context_size": len(sources),
        "hit": hit,
        "recall": recall,
        "first_relevant_rank": first_relevant_rank,
        "reciprocal_rank": reciprocal_rank,
        "ndcg": ndcg,
        "chunk_match_rate": chunk_match_rate,
    }

In [8]:
# Увеличиваем допустимую ширину текста в таблицах.
pd.set_option("display.max_colwidth", 250)


for number, answer_record in enumerate(answers, start=1):
    query_id = str(answer_record["query_id"])
    query_text = answer_record["query_text"]
    generated_answer = answer_record["answer"]
    full_prompt = answer_record["prompt"]

    context_metrics = calculate_context_metrics(answer_record)

    display(
        HTML(
            f"<h2>Пример {number}. Query ID: {escape(query_id)}</h2>"
        )
    )

    # Выводим исходное научное утверждение из SciFact.
    display(Markdown("### Исходный запрос"))

    display(
        HTML(
            f"<div style='padding: 10px; "
            f"border-left: 4px solid #4c78a8;'>"
            f"{escape(query_text)}"
            f"</div>"
        )
    )

    # Полный prompt достаточно большой, поэтому помещаем его
    # в сворачиваемый блок.
    display(
        HTML(
            "<details>"
            "<summary><b>Показать полный prompt, отправленный GigaChat</b></summary>"
            f"<pre style='white-space: pre-wrap;'>"
            f"{escape(full_prompt)}"
            "</pre>"
            "</details>"
        )
    )

    # Выводим сохранённый ответ модели.
    display(Markdown("### Ответ GigaChat"))

    display(
        HTML(
            f"<div style='padding: 10px; "
            f"border-left: 4px solid #59a14f;'>"
            f"<pre style='white-space: pre-wrap; "
            f"font-family: inherit;'>"
            f"{escape(generated_answer)}"
            f"</pre>"
            f"</div>"
        )
    )

    display(Markdown("### Эталон — наш вариант"))

    # Показываем рядом эталонные документы и найденные системой чанки.
    comparison_table = pd.DataFrame(
        [
            {
                "Эталонные doc_id из SciFact": ", ".join(
                    context_metrics["gold_documents"]
                ),
                "Doc_id найденных чанков": ", ".join(
                    context_metrics["source_document_ids"]
                ),
                "Совпавшие doc_id": (
                    ", ".join(context_metrics["matched_documents"])
                    if context_metrics["matched_documents"]
                    else "Нет совпадений"
                ),
            }
        ]
    )

    display(
        comparison_table.style.hide(axis="index")
    )

    # Формируем подробную таблицу найденных чанков.
    chunk_rows = []

    for source in context_metrics["sources"]:
        document_id = str(source.get("doc_id"))
        chunk_text = " ".join(
            str(source.get("text", "")).split()
        )

        # Сокращаем текст только для удобства отображения.
        # Полный текст остаётся в JSONL и в полном prompt.
        if len(chunk_text) > 300:
            chunk_text = chunk_text[:300] + "…"

        chunk_rows.append(
            {
                "Ранг": source.get("retrieval_rank"),
                "Источник": f'[{source.get("source_number")}]',
                "Chunk ID": source.get("chunk_id"),
                "Doc ID": document_id,
                "Retrieval score": source.get("retrieval_score"),
                "Соответствует эталону": (
                    "Да"
                    if document_id
                    in context_metrics["gold_documents"]
                    else "Нет"
                ),
                "Фрагмент чанка": chunk_text,
            }
        )

    chunks_table = pd.DataFrame(chunk_rows)

    display(Markdown("### Найденные чанки"))

    display(
    chunks_table.style
    .format({"Retrieval score": "{:.4f}"})
    .map(
        lambda value: (
            "background-color: #b7e1cd"
            if value == "Да"
            else ""
        ),
        subset=["Соответствует эталону"],
    )
    .hide(axis="index")
)

    # Выводим рассчитанные retrieval-метрики.
    context_size = context_metrics["context_size"]

    metrics_table = pd.DataFrame(
        [
            {
                "Метрика": f"Hit@{context_size}",
                "Значение": f'{context_metrics["hit"]:.0f}',
            },
            {
                "Метрика": f"Recall@{context_size}",
                "Значение": f'{context_metrics["recall"]:.4f}',
            },
            {
                "Метрика": f"RR@{context_size}",
                "Значение": (
                    f'{context_metrics["reciprocal_rank"]:.4f}'
                ),
            },
            {
                "Метрика": f"nDCG@{context_size}",
                "Значение": f'{context_metrics["ndcg"]:.4f}',
            },
            {
                "Метрика": "Доля подходящих чанков",
                "Значение": (
                    f'{context_metrics["chunk_match_rate"]:.4f}'
                ),
            },
            {
                "Метрика": "Ранг первого совпадения",
                "Значение": (
                    str(context_metrics["first_relevant_rank"])
                    if context_metrics["first_relevant_rank"]
                    is not None
                    else "Эталонный документ не найден"
                ),
            },
        ]
    )

    display(Markdown("### Метрики соответствия контекста"))

    display(
        metrics_table.style.hide(axis="index")
    )

    # Добавляем результаты ручной оценки самого ответа.
    if query_id in evaluation_by_query.index:
        manual_result = evaluation_by_query.loc[query_id]

        answer_metrics_table = pd.DataFrame(
            [
                {
                    "Метрика ответа": "Корректность",
                    "Оценка": (
                        f'{manual_result["correctness_0_2"]:.0f} / 2'
                    ),
                },
                {
                    "Метрика ответа": "Обоснованность источниками",
                    "Оценка": (
                        f'{manual_result["groundedness_0_2"]:.0f} / 2'
                    ),
                },
                {
                    "Метрика ответа": "Полнота",
                    "Оценка": (
                        f'{manual_result["completeness_0_2"]:.0f} / 2'
                    ),
                },
                {
                    "Метрика ответа": "Качество цитирования",
                    "Оценка": (
                        f'{manual_result["citation_quality_0_2"]:.0f} / 2'
                    ),
                },
                {
                    "Метрика ответа": "Тип ошибки",
                    "Оценка": manual_result["error_type"],
                },
                {
                    "Метрика ответа": "Комментарий",
                    "Оценка": manual_result["notes"],
                },
            ]
        )

        display(Markdown("### Ручная оценка ответа"))

        display(
            answer_metrics_table.style.hide(axis="index")
        )

    display(HTML("<hr style='margin: 35px 0;'>"))

### Исходный запрос

### Ответ GigaChat

### Эталон — наш вариант

Эталонные doc_id из SciFact,Doc_id найденных чанков,Совпавшие doc_id
13768432,"13768432, 38040186, 5597586, 32671519",13768432


### Найденные чанки

Ранг,Источник,Chunk ID,Doc ID,Retrieval score,Соответствует эталону,Фрагмент чанка
1,[1],13768432:recursive:0002,13768432,0.8331,Да,"##ricular mass index ( hazard ratio, 4. 1 ; 95 % confidence interval, 1. 3 - 13. 1 ; p < 0. 05 ). CONCLUSIONS There is a continuum of cardiac involvement in systemic AL and ATTR amyloidosis. Transmural LGE is determined reliably by PSIR and represents advanced cardiac amyloidosis. The PSIR technique…"
2,[2],38040186:recursive:0001,38040186,0.7296,Нет,"patients with primary amyloidosis. when patients were stratified by organ involvement, the survival of patients with familial amyloidosis remained superior. The presence of cardiomyopathy and an interactive variable of age and the presence of autonomic neuropathy were powerful predictors of survival…"
3,[3],5597586:recursive:0001,5597586,0.7098,Нет,"without an ( p = 0. 0015 ). median duration of hospitalization was similar for both cohorts, as were engraftment kinetics. Atrial fibrillation occurred in all patients with AN but in only 1 control patient (p < 0.0001). Median overall survival (OS) for patients with AN was 29 months but >60 months f…"
4,[4],32671519:recursive:0001,32671519,0.7051,Нет,"identify the extent of amyloid deposition in all 50 patients. there was no uptake of the 123i - sap by the control patients and the healthy subjects. In all patients with AA amyloidosis the spleen was affected, whereas the scans showed uptake in the heart, skin, carpal region, and bone marrow only i…"


... (70 outputs left out)

### Метрики соответствия контекста

Метрика,Значение
Hit@4,1
Recall@4,1.0000
RR@4,1.0000
nDCG@4,1.0000
Доля подходящих чанков,0.5000
Ранг первого совпадения,1


### Ручная оценка ответа

Метрика ответа,Оценка
Корректность,2 / 2
Обоснованность источниками,2 / 2
Полнота,2 / 2
Качество цитирования,1 / 2
Тип ошибки,format_noncompliance
Комментарий,Вывод напрямую поддержан источником 1; отсутствует итоговая строка ИСТОЧНИКИ.
